In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import cross_val_score
import seaborn as sns

In [10]:
CarDF = pd.read_csv('./data/Dubizzle_used_car_sales.csv')
CarDF.head()

,title,price_in_aed,kilometers,body_condition,mechanical_condition,seller_type,body_type,no_of_cylinders,transmission_type,regional_specs,horsepower,fuel_type,steering_side,year,color,emirate,motors_trim,company,model,date_posted
0,MITSUBISHI PAJERO 3.5L / 2013,26000,167390,Perfect inside and out,Perfect inside and out,Dealer,SUV,6,Automatic Transmission,GCC Specs,Unknown,Gasoline,Left Hand Side,2013.0,Silver,Dubai,GLS,mitsubishi,pajero,13/05/2022
1,chevrolet silverado,110000,39000,Perfect inside and out,Perfect inside and out,Dealer,SUV,8,Automatic Transmission,North American Specs,400 - 500 HP,Gasoline,Left Hand Side,2018.0,White,Sharjah,1500 High Country,chevrolet,silverado,14/01/2022
2,MERCEDES-BENZ E300 - 2014 - GCC SPEC - FULL OP...,78000,200000,Perfect inside and out,Perfect inside and out,Dealer,Sedan,6,Automatic Transmission,GCC Specs,400 - 500 HP,Gasoline,Left Hand Side,2014.0,Blue,Sharjah,E 300,mercedes-benz,e-class,05/05/2022
3,WARRANTY UNTIL APR 2023 || Ferrari 488 Spider ...,899000,27000,Perfect inside and out,Perfect inside and out,Dealer,Hard Top Convertible,8,Automatic Transmission,GCC Specs,600 - 700 HP,Gasoline,Left Hand Side,2018.0,Red,Dubai,Standard,ferrari,488-spider,30/04/2022
4,USED RENAULT DOKKER 2020,33000,69000,Perfect inside and out,Perfect inside and out,Owner,Wagon,4,Manual Transmission,GCC Specs,Less than 150 HP,Gasoline,Left Hand Side,2020.0,White,Dubai,Standard,renault,dokker,13/05/2022


In [11]:
CarDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9970 entries, 0 to 9969
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   title                 9965 non-null   object 
 1   price_in_aed          9970 non-null   int64  
 2   kilometers            9970 non-null   int64  
 3   body_condition        9970 non-null   object 
 4   mechanical_condition  9970 non-null   object 
 5   seller_type           9970 non-null   object 
 6   body_type             9970 non-null   object 
 7   no_of_cylinders       9889 non-null   object 
 8   transmission_type     9970 non-null   object 
 9   regional_specs        9970 non-null   object 
 10  horsepower            9970 non-null   object 
 11  fuel_type             9970 non-null   object 
 12  steering_side         9970 non-null   object 
 13  year                  9000 non-null   float64
 14  color                 9970 non-null   object 
 15  emirate              

| 변수명                      | 결측치        | 설명                      | 처리 방향                        |
| ------------------------ | ---------- | ----------------------- | ---------------------------- |
| ✅ `price_in_aed`         | 없음         | 🎯 예측할 **자동차 가격 (AED)** | **타깃 변수**                    |
| ✅ `kilometers`           | 없음         | 누적 주행 거리 (km)           | 수치형 그대로 사용                   |
| ⚠️ `year`                | ✅ 결측치 970개 | 제조 연도 (float이지만 연도임)    | 결측치 제거 또는 보간                 |
| ✅ `body_condition`       | 없음         | 외관 상태                   | **LabelEncoding 또는 One-Hot** |
| ✅ `mechanical_condition` | 없음         | 기계 상태                   | 인코딩                          |
| ✅ `seller_type`          | 없음         | 판매자 유형 (딜러/소유자)         | 인코딩                          |
| ✅ `body_type`            | 없음         | 차량 형태 (SUV, 세단 등)       | 인코딩                          |
| ⚠️ `no_of_cylinders`     | ✅ 결측치 81개  | 엔진 실린더 수 (`object`)     | 문자열을 숫자로 변환 or 인코딩           |
| ✅ `transmission_type`    | 없음         | 변속기 종류                  | 인코딩                          |
| ✅ `regional_specs`       | 없음         | 중동 사양 등                 | 인코딩                          |
| ✅ `horsepower`           | 없음         | 마력 범위 (`object`)        | 범위 처리 필요 (예: 중앙값 추출)         |
| ✅ `fuel_type`            | 없음         | 연료 종류                   | 인코딩                          |
| ✅ `emirate`              | 없음         | 등록 지역 (두바이 등)           | 인코딩                          |
| ✅ `company`              | 없음         | 브랜드명                    | 인코딩                          |
| ✅ `model`                | 없음         | 차량 모델명                  | 인코딩 또는 Top N만 사용             |
| ⚠️ `steering_side`       | 없음         | 대부분 ‘Left Hand Side’    | **변수 정보가 거의 없으면 제외 가능**      |
| ⚠️ `title`               | ✅ 결측치 5개   | 게시글 제목 (문장)             | 제외하거나 NLP 분석                 |
| ⚠️ `motors_trim`         | ✅ 결측치 28개  | 상세 트림명                  | 너무 다양하면 제거                   |
| ⚠️ `color`               | 없음         | 차량 색상                   | 영향 미미 → 제거 가능                |
| ⚠️ `date_posted`         | 없음         | 게시 날짜                   | 예측에는 직접 영향 없음 → 제거 가능        |


In [ ]:
## 모든 조건이 동일한 차량에 대해 모든 색상은 가격이 동일하다

In [22]:
# motors_trim 변수의 고유값 개수
unique_count = CarDF['motors_trim'].nunique()
print(f"'motors_trim' 고유값 개수: {unique_count}")

# 고유값 샘플 20개 출력
unique_values_sample = CarDF['motors_trim'].dropna().unique()[:20]
print("motors_trim 샘플 값 20개:", unique_values_sample)

## 상위 빈도 몇 개만 살리고 나머지는 'Other'로 묶거나 중요도가 낮으면 제외


'motors_trim' 고유값 개수: 856
motors_trim 샘플 값 20개: ['GLS' '1500 High Country' 'E 300' 'Standard' 'S-line' 'SRT' 'Other'
 '70th Anniversary' 'SV' 'G 63 AMG' 'Limited' 'Sport' 'Veloce' 'SE' '630i'
 '3.8 V6 4WD' 'Super' 'CL 63 AMG' 'Adventure' '640i']


In [23]:
# title, steering_side, color, date_posted 컬럼을 제거
CarDF = CarDF.drop(columns=['title', 'steering_side', 'color', 'date_posted'])

In [24]:
CarDF.head()

,price_in_aed,kilometers,body_condition,mechanical_condition,seller_type,body_type,no_of_cylinders,transmission_type,regional_specs,horsepower,fuel_type,year,emirate,motors_trim,company,model
0,26000,167390,Perfect inside and out,Perfect inside and out,Dealer,SUV,6,Automatic Transmission,GCC Specs,Unknown,Gasoline,2013.0,Dubai,GLS,mitsubishi,pajero
1,110000,39000,Perfect inside and out,Perfect inside and out,Dealer,SUV,8,Automatic Transmission,North American Specs,400 - 500 HP,Gasoline,2018.0,Sharjah,1500 High Country,chevrolet,silverado
2,78000,200000,Perfect inside and out,Perfect inside and out,Dealer,Sedan,6,Automatic Transmission,GCC Specs,400 - 500 HP,Gasoline,2014.0,Sharjah,E 300,mercedes-benz,e-class
3,899000,27000,Perfect inside and out,Perfect inside and out,Dealer,Hard Top Convertible,8,Automatic Transmission,GCC Specs,600 - 700 HP,Gasoline,2018.0,Dubai,Standard,ferrari,488-spider
4,33000,69000,Perfect inside and out,Perfect inside and out,Owner,Wagon,4,Manual Transmission,GCC Specs,Less than 150 HP,Gasoline,2020.0,Dubai,Standard,renault,dokker


In [26]:
# 컬럼별 결측치 개수 확인
missing_counts = CarDF.isnull().sum()
print(missing_counts)
print('------------------')
# 결측치가 있는 컬럼만 보기 -no_of_cylinders -year -  motors_trim   
missing_cols = missing_counts[missing_counts > 0]
print(missing_cols)


price_in_aed              0
kilometers                0
body_condition            0
mechanical_condition      0
seller_type               0
body_type                 0
no_of_cylinders          81
transmission_type         0
regional_specs            0
horsepower                0
fuel_type                 0
year                    970
emirate                   0
motors_trim              28
company                   0
model                     0
dtype: int64
------------------
no_of_cylinders     81
year               970
motors_trim         28
dtype: int64
